In [33]:
from langchain_core.messages.tool import tool_call
from langchain_ollama import ChatOllama
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
import requests

In [43]:
from langchain_core.tools import InjectedToolArg
from typing import Annotated
@tool
def get_conversion_factor(base_currency: str, target_currency: str) -> float:
    """This function fetches the currency conversion factor between a given base currency and a target currency"""

    url = f"https://v6.exchangerate-api.com/v6/ec2702bf9c0a72c464a57cb4/pair/{base_currency}/{target_currency}"

    response=requests.get(url)
    return response.json()

@tool
def convert(base_currency_value:int,conversion_rate:Annotated[float,InjectedToolArg])->float:
    """Given a currency conversion rate this fucntion calculates the target currency value from a given basee currency value"""

    return base_currency_value*conversion_rate

In [35]:
llm = ChatOllama(model="llama3.1")

In [36]:
llm_with_tools=llm.bind_tools([get_conversion_factor,convert])

In [37]:
messages=[HumanMessage("what is the conversion factor between USD and INR and based on that you can connvert 10 usd to inr ")]

In [38]:
messages

[HumanMessage(content='what is the conversion factor between USD and INR and based on that you can connvert 10 usd to inr ', additional_kwargs={}, response_metadata={})]

In [39]:
ai_messages=llm_with_tools.invoke(messages)

In [40]:
messages.append(ai_messages)

In [41]:
ai_messages

AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'llama3.1', 'created_at': '2026-07-06T07:18:36.1467954Z', 'done': True, 'done_reason': 'stop', 'total_duration': 6198273000, 'load_duration': 5021526700, 'prompt_eval_count': 261, 'prompt_eval_duration': 217295000, 'eval_count': 44, 'eval_duration': 949341000, 'logprobs': None, 'model_name': 'llama3.1', 'model_provider': 'ollama'}, id='lc_run--019f364a-f53a-7c60-8772-a729468b5500-0', tool_calls=[{'name': 'get_conversion_factor', 'args': {'base_currency': 'USD', 'target_currency': 'INR'}, 'id': 'd68d4d6b-b4d1-4c19-891c-6b8a9d9ddbf6', 'type': 'tool_call'}, {'name': 'convert', 'args': {'base_currency_value': '10'}, 'id': '184fd823-6b47-4548-ae90-4eb504b43a59', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 261, 'output_tokens': 44, 'total_tokens': 305})

In [45]:
import json
for tool_call in ai_messages.tool_calls:
    if tool_call['name']=="get_conversion_factor":
        tool_message1=get_conversion_factor.invoke(tool_call)
        conversion_rate=json.loads(tool_message1.content)["conversion_rate"]
        messages.append(tool_message1)
    if tool_call['name']=='convert':
        tool_call['args']['conversion_rate']=conversion_rate
        tool_message2=convert.invoke(tool_call)
        messages.append(tool_message2)


In [48]:
messages

[HumanMessage(content='what is the conversion factor between USD and INR and based on that you can connvert 10 usd to inr ', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'llama3.1', 'created_at': '2026-07-06T07:18:36.1467954Z', 'done': True, 'done_reason': 'stop', 'total_duration': 6198273000, 'load_duration': 5021526700, 'prompt_eval_count': 261, 'prompt_eval_duration': 217295000, 'eval_count': 44, 'eval_duration': 949341000, 'logprobs': None, 'model_name': 'llama3.1', 'model_provider': 'ollama'}, id='lc_run--019f364a-f53a-7c60-8772-a729468b5500-0', tool_calls=[{'name': 'get_conversion_factor', 'args': {'base_currency': 'USD', 'target_currency': 'INR'}, 'id': 'd68d4d6b-b4d1-4c19-891c-6b8a9d9ddbf6', 'type': 'tool_call'}, {'name': 'convert', 'args': {'base_currency_value': '10', 'conversion_rate': 95.3348}, 'id': '184fd823-6b47-4548-ae90-4eb504b43a59', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'